# 03 Model Bias and Overfit Analysis

This notebook inspects train/validation/test behavior using saved report files and rolling CV outputs.

In [1]:
from pathlib import Path
import pandas as pd

REPORTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports')

In [2]:
metrics = pd.concat([
    pd.read_csv(REPORTS_DIR / 'global_models_a_metrics.csv'),
    pd.read_csv(REPORTS_DIR / 'global_models_b_metrics.csv'),
    pd.read_csv(REPORTS_DIR / 'global_models_c_metrics.csv'),
], ignore_index=True)
metrics.head()

,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,A,XGBOOST,val,1,618,0.121885,5355.007381,-1897.957357,1.036329,1872.075360,0.684466
1,A,XGBOOST,val,2,618,0.130412,5737.973812,-1874.453600,1.093050,2003.044340,0.619741
2,A,XGBOOST,val,3,618,0.138934,6274.890377,-2641.075425,1.171659,2133.932841,0.708738
3,A,XGBOOST,val,4,618,0.134269,5911.844895,-2266.316214,1.152634,2062.284881,0.684466
4,A,XGBOOST,val,5,618,0.134136,5963.619276,-2403.847959,1.158810,2060.235332,0.694175


## Overall Test Ranking

In [3]:
metrics[(metrics['split']=='test') & (metrics['horizon']==0)].sort_values(['dataset','WAPE','MASE_mean','RMSE'])

,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
51,A,CATBOOST,test,0,14832,0.116281,5337.523418,-1238.852308,1.038602,1795.501209,0.548139
25,A,XGBOOST,test,0,14832,0.117375,5378.519153,-1040.790931,1.032125,1812.390244,0.573894
77,B,XGBOOST,test,0,14832,0.116523,5349.681648,-1065.374571,1.025741,1799.240847,0.569512
103,B,CATBOOST,test,0,14832,0.119611,5499.270616,-1244.578790,1.084185,1846.925958,0.549083
155,C,CATBOOST,test,0,222480,0.180280,8528.362570,-1329.265066,1.099196,2744.921854,0.525094
129,C,XGBOOST,test,0,222480,0.184818,8730.396722,-1123.000460,1.089718,2814.026574,0.528969


## Bias Review

In [4]:
metrics[(metrics['split']=='test') & (metrics['horizon']==0)][['dataset','model','Bias','under_forecast_rate']].sort_values(['dataset','Bias'])

,dataset,model,Bias,under_forecast_rate
51,A,CATBOOST,-1238.852308,0.548139
25,A,XGBOOST,-1040.790931,0.573894
103,B,CATBOOST,-1244.578790,0.549083
77,B,XGBOOST,-1065.374571,0.569512
155,C,CATBOOST,-1329.265066,0.525094
129,C,XGBOOST,-1123.000460,0.528969


## Horizon Drift

In [5]:
metrics[(metrics['split']=='test') & (metrics['horizon'] > 0)][['dataset','model','horizon','WAPE','MASE_mean','RMSE']].sort_values(['dataset','model','horizon']).head(50)

,dataset,model,horizon,WAPE,MASE_mean,RMSE
39,A,CATBOOST,1,0.110144,0.991505,5076.558607
40,A,CATBOOST,2,0.115816,1.048508,5346.613477
41,A,CATBOOST,3,0.122155,1.051432,5651.137133
42,A,CATBOOST,4,0.124635,1.069758,5741.051018
43,A,CATBOOST,5,0.129700,1.113238,6017.969664
44,A,CATBOOST,6,0.106970,0.985479,4954.358626
45,A,CATBOOST,7,0.112861,1.037890,5082.807806
46,A,CATBOOST,8,0.115706,1.151978,5011.535560
47,A,CATBOOST,9,0.108831,0.962274,4998.099576
48,A,CATBOOST,10,0.108722,0.948849,5097.389350


## Rolling CV Summary

Run `rolling_cv.py` before this cell if the summary file does not exist.

In [6]:
cv_path = REPORTS_DIR / 'rolling_cv_metrics_summary.csv'
if cv_path.exists():
    pd.read_csv(cv_path).head(30)
else:
    print('Missing rolling_cv_metrics_summary.csv. Run rolling_cv.py first.')

Missing rolling_cv_metrics_summary.csv. Run rolling_cv.py first.
